In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
np.random.seed(12);tf.random.set_seed(12)
(x,y),(xt,yt)=tf.keras.datasets.cifar10.load_data();x=x[:10000].astype("float32")/255.;y=y[:10000].ravel();xt=xt[:2000].astype("float32")/255.;yt=yt[:2000].ravel()
results=[]
for convs in [2,3,4]:
 for kernel in [3,5]:
  for pool in ["max","average"]:
   for batchnorm in [False,True]:
    for lr in [.001,.0003]:
     inp=tf.keras.Input((32,32,3));z=inp
     for j in range(convs):
      z=tf.keras.layers.Conv2D(16*(2**min(j,2)),kernel,padding="same",use_bias=not batchnorm)(z)
      if batchnorm:z=tf.keras.layers.BatchNormalization()(z)
      z=tf.keras.layers.ReLU()(z)
      if j%2==1:z=tf.keras.layers.MaxPooling2D()(z) if pool=="max" else tf.keras.layers.AveragePooling2D()(z)
     z=tf.keras.layers.GlobalAveragePooling2D()(z);out=tf.keras.layers.Dense(10,activation="softmax")(z);m=tf.keras.Model(inp,out)
     m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
     start=time.time();m.fit(x,y,epochs=2,batch_size=64,validation_split=.1,verbose=0);elapsed=time.time()-start
     acc=m.evaluate(xt,yt,verbose=0)[1]
     results.append([convs,kernel,pool,batchnorm,lr,64,m.count_params(),acc,elapsed])
df=pd.DataFrame(results,columns=["Conv layers","Filter size","Pooling","Batch norm","Learning rate","Batch size","Parameters","Test accuracy","Training seconds"])
print(df.round(4).to_string(index=False));print("\nBest configuration:\n",df.loc[df["Test accuracy"].idxmax()].to_string())
fig,ax=plt.subplots(1,3,figsize=(15,4));ax[0].scatter(df["Parameters"],df["Test accuracy"]);ax[0].set_xlabel("Parameters");ax[0].set_ylabel("Accuracy");ax[0].set_title("Accuracy vs model size");ax[1].scatter(df["Training seconds"],df["Test accuracy"]);ax[1].set_xlabel("Training seconds");ax[1].set_title("Accuracy vs training time");ax[2].bar(range(len(df)),df["Test accuracy"]);ax[2].set_title("Configuration accuracy");plt.tight_layout();plt.show()